# Prepare Covertype data

This notebook reads the extracted API rows from `cover_samples`, creates stratified training, validation, and test splits, and writes them to the new `cover_prepared` PostgreSQL table. No local CSV or cloud bucket is used for dataset preparation.

Run it after `extract_cover_data` has populated the source table. Re-running the load cell replaces the contents of `cover_prepared`.

In [2]:
import os
import pandas as pd
import psycopg
from psycopg.rows import dict_row
from sklearn.model_selection import train_test_split

DB_CONFIG = {
    "host": os.getenv("PROJECT_DATA_DB_HOST", "localhost"),
    "port": int(os.getenv("PROJECT_DATA_DB_PORT", "5433")),
    "dbname": os.getenv("PROJECT_DATA_DB_NAME", "cover_data"),
    "user": os.getenv("PROJECT_DATA_DB_USER", "cover_user"),
    "password": os.getenv("PROJECT_DATA_DB_PASSWORD", "cover_password"),
}
SOURCE_TABLE = "cover_samples"
PREPARED_TABLE = "cover_prepared"
SEED = 42


## Load extracted rows

The source table stores each Covertype field as text. Convert the ten measured features and target to numbers, keep wilderness area and soil type as categories, and retain group and batch provenance.

In [3]:
NUMERIC_FEATURES = [
    "elevation", "aspect", "slope", "horizontal_distance_to_hydrology",
    "vertical_distance_to_hydrology", "horizontal_distance_to_roadways",
    "hillshade_9am", "hillshade_noon", "hillshade_3pm",
    "horizontal_distance_to_fire_points",
]
CATEGORICAL_FEATURES = ["wilderness_area", "soil_type"]
TARGET = "cover_type"
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

with psycopg.connect(**DB_CONFIG, row_factory=dict_row) as conn:
    source = pd.DataFrame(conn.execute(
        f"SELECT group_number, batch_number, {', '.join(FEATURES)}, {TARGET} FROM {SOURCE_TABLE}"
    ).fetchall())

if source.empty:
    raise RuntimeError("cover_samples is empty. Run the extraction DAG before preparing data.")

for column in NUMERIC_FEATURES + [TARGET]:
    source[column] = pd.to_numeric(source[column], errors="raise")
source[TARGET] = source[TARGET].astype("int16")
cover_types = set(source[TARGET].unique().tolist())
if 0 in cover_types and cover_types.issubset(set(range(7))):
    # The API labels classes 0-6; the model/API contract uses Covertype's 1-7 labels.
    source[TARGET] = source[TARGET] + 1
elif not cover_types.issubset(set(range(1, 8))):
    raise ValueError(f"Unexpected cover_type labels: {sorted(cover_types)}")
for column in CATEGORICAL_FEATURES:
    source[column] = source[column].astype(str)

rows_before_deduplication = len(source)
source = source.drop_duplicates(subset=FEATURES + [TARGET]).reset_index(drop=True)
duplicates_removed = rows_before_deduplication - len(source)

print(f"Loaded {rows_before_deduplication:,} rows from {SOURCE_TABLE}.")
print(f"Removed {duplicates_removed:,} repeated records across batches.")
print(f"Rows available for stratified splits: {len(source):,}.")
print("Cover type counts:")
display(source[TARGET].value_counts().sort_index().rename_axis(TARGET).to_frame("rows"))


Loaded 580,000 rows from cover_samples.
Removed 521,899 repeated records across batches.
Rows available for stratified splits: 58,101.
Cover type counts:


,rows
cover_type,
1,11937
2,21535
3,15146
4,547
5,1889
6,5790
7,1257


## Make stratified splits

Keep every cover type represented in each split. The split proportions are 70% train, 15% validation, and 15% test.

In [4]:
train, remaining = train_test_split(
    source, test_size=0.30, random_state=SEED, stratify=source[TARGET]
)
validation, test = train_test_split(
    remaining, test_size=0.50, random_state=SEED, stratify=remaining[TARGET]
)

train = train.assign(split="train")
validation = validation.assign(split="validation")
test = test.assign(split="test")
prepared = pd.concat([train, validation, test], ignore_index=True)
print(prepared["split"].value_counts())
display(prepared.head())


split
train         40670
test           8716
validation     8715
Name: count, dtype: int64


,group_number,batch_number,elevation,aspect,slope,horizontal_distance_to_hydrology,vertical_distance_to_hydrology,horizontal_distance_to_roadways,hillshade_9am,hillshade_noon,hillshade_3pm,horizontal_distance_to_fire_points,wilderness_area,soil_type,cover_type,split
0,4,1,2059,34,30,127,109,150,200,160,78,170,Cache,C4703,3,train
1,4,1,2100,303,25,60,32,631,141,218,213,180,Cache,C4703,3,train
2,4,1,2869,58,22,170,64,192,230,188,80,430,Rawah,C7745,1,train
3,4,1,2003,315,34,124,97,212,111,190,208,424,Cache,C4703,3,train
4,4,1,2167,295,37,120,90,1405,97,203,234,888,Cache,C4703,3,train


## Create and populate `cover_prepared`

The table keeps source group and batch for traceability, along with the 12 model features, label, and split name. The generated primary key identifies rows. This cell truncates and reloads only the prepared table; it leaves `cover_samples` unchanged.

In [5]:
CREATE_PREPARED_SQL = f"""
CREATE TABLE IF NOT EXISTS {PREPARED_TABLE} (
    prepared_id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    group_number INTEGER NOT NULL,
    batch_number INTEGER NOT NULL,
    elevation INTEGER NOT NULL,
    aspect INTEGER NOT NULL,
    slope INTEGER NOT NULL,
    horizontal_distance_to_hydrology INTEGER NOT NULL,
    vertical_distance_to_hydrology INTEGER NOT NULL,
    horizontal_distance_to_roadways INTEGER NOT NULL,
    hillshade_9am INTEGER NOT NULL,
    hillshade_noon INTEGER NOT NULL,
    hillshade_3pm INTEGER NOT NULL,
    horizontal_distance_to_fire_points INTEGER NOT NULL,
    wilderness_area TEXT NOT NULL,
    soil_type TEXT NOT NULL,
    cover_type SMALLINT NOT NULL CHECK (cover_type BETWEEN 1 AND 7),
    split TEXT NOT NULL CHECK (split IN ('train', 'validation', 'test'))
)
"""
DB_COLUMNS = ["group_number", "batch_number", *FEATURES, TARGET, "split"]
INSERT_SQL = f"""INSERT INTO {PREPARED_TABLE}
    ({', '.join(DB_COLUMNS)}) VALUES ({', '.join(['%s'] * len(DB_COLUMNS))})"""

with psycopg.connect(**DB_CONFIG) as conn, conn.cursor() as cursor:
    cursor.execute(CREATE_PREPARED_SQL)
    cursor.execute(f"TRUNCATE TABLE {PREPARED_TABLE}")
    records = (
        tuple(value.item() if hasattr(value, "item") else value for value in row)
        for row in prepared[DB_COLUMNS].itertuples(index=False, name=None)
    )
    cursor.executemany(INSERT_SQL, records)

print(f"Loaded {len(prepared):,} rows into {PREPARED_TABLE}.")


Loaded 58,101 rows into cover_prepared.


## Inspect prepared data

In [6]:
with psycopg.connect(**DB_CONFIG, row_factory=dict_row) as conn:
    summary = pd.DataFrame(conn.execute(
        f"SELECT split, COUNT(*) AS rows FROM {PREPARED_TABLE} GROUP BY split ORDER BY split"
    ).fetchall())
    head = pd.DataFrame(conn.execute(
        f"SELECT {', '.join(DB_COLUMNS)} FROM {PREPARED_TABLE} ORDER BY prepared_id LIMIT 5"
    ).fetchall())
print("Prepared rows by split:")
display(summary)
print("Prepared table head:")
display(head)


Prepared rows by split:


,split,rows
0,test,8716
1,train,40670
2,validation,8715


Prepared table head:


,group_number,batch_number,elevation,aspect,slope,horizontal_distance_to_hydrology,vertical_distance_to_hydrology,horizontal_distance_to_roadways,hillshade_9am,hillshade_noon,hillshade_3pm,horizontal_distance_to_fire_points,wilderness_area,soil_type,cover_type,split
0,4,1,2059,34,30,127,109,150,200,160,78,170,Cache,C4703,3,train
1,4,1,2100,303,25,60,32,631,141,218,213,180,Cache,C4703,3,train
2,4,1,2869,58,22,170,64,192,230,188,80,430,Rawah,C7745,1,train
3,4,1,2003,315,34,124,97,212,111,190,208,424,Cache,C4703,3,train
4,4,1,2167,295,37,120,90,1405,97,203,234,888,Cache,C4703,3,train
